# georeg - registration experiments
Runs on synthetic data with exact ground truth. Execute from the project root (after `pip install -e .`).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from pathlib import Path
from georeg.synthetic_transform import make_reference_geotiff, make_case, PRESETS
from georeg.dataset_loader import load_raster, prepare_gray, detection_mask, describe_raster
from georeg.config import PipelineCfg, DetectorCfg, MatcherCfg
from georeg.geotiff_writer import write_geotiff

work = Path('data/notebook'); work.mkdir(parents=True, exist_ok=True)
make_reference_geotiff(work/'reference.tif', 768, seed=7)
ref = load_raster(work/'reference.tif')
print(describe_raster(ref)['crs'], ref.width, ref.height, ref.transform)

## 1. Make a test case with known truth

In [ ]:
case = make_case(ref.data, seed=3, **PRESETS['affine'])
write_geotiff(work/'source.tif', case.data, None, None, valid=case.valid, nodata=0)
print('truth T (src->ref):'); print(np.round(case.T_true, 5))

## 2. Features and matches (SIFT vs ORB)

In [ ]:
from georeg.feature_detector import detect_features
from georeg.feature_matcher import match_features
from georeg.transformation_estimator import reprojection_errors
pre = PipelineCfg().preprocess
sg, rg = prepare_gray(case.data, case.valid, pre), prepare_gray(ref.data, None, pre)
for m in ('sift','orb'):
    d = DetectorCfg(m)
    mr = match_features(detect_features(sg, d, detection_mask(case.valid)), detect_features(rg, d), MatcherCfg())
    right = (reprojection_errors(case.T_true, mr.pts_src, mr.pts_ref) < 3).mean()
    print(m, 'matches', len(mr), 'geometrically correct: %.0f%%' % (100*right))

## 3. Full pipeline with ground-truth scoring

In [ ]:
from georeg.pipeline import run_registration
res = run_registration(work/'source.tif', work/'reference.tif', work/'out', PipelineCfg(), run_name='nb', T_true=case.T_true)
print('accepted:', res.success, '| model:', res.report['model'])
g = res.report['ground_truth']; print('GT RMSE %.3f px = %.2f m' % (g['rmse_px'], g['rmse_map']))
print('naive stamp error: %.1f px' % res.report['naive_stamp_ground_truth']['rmse_px'])

In [ ]:
img = plt.imread(res.outputs['before_after']); plt.figure(figsize=(14,7)); plt.imshow(img); plt.axis('off'); plt.show()

## 4. Sweep one factor yourself

In [ ]:
from georeg.robustness import run_robustness
out = run_robustness(work/'robust', size=512, n_trials=1, factors={'noise sigma': [0, 40, 100], 'rotation_deg': [0, 90]})
for s in out['summary']: print(s['factor'], s['level'], s['detector'], '%.0f%%' % (100*s['success_rate']))